# **ConSurf-AI**: per-residue evolutionary conservation from a single structure

ConSurf-AI predicts ConSurf conservation grades (**1** = variable … **9** = conserved) for every residue of a protein chain, without building a multiple sequence alignment. It runs a small attention network on **ESM2-3B** embeddings.

**How to use**
1. `Runtime` → `Change runtime type` → select a **GPU** (T4 is enough).
2. Run the cells in order (or `Runtime` → `Run all`). Cell **0** installs the packages and downloads the model weights once (ESM2-3B is ~11 GB, so allow a few minutes). Cells **1** and **2** let you pick the structure and chain. You can change them and rerun cells 1–5 without rerunning cell 0.

**Colour key.** These are the standard ConSurf colours. Yellow marks residues the model is uncertain about (predicted uncertainty ≥ 2). This plays the same role as ConSurf's "insufficient data" flag.

In [ ]:
#@title 0. Install & initialise (run first) {display-mode: "form"}
#@markdown Installs the extra package (gemmi), defines the ConSurf-AI helpers and downloads the model weights (ConSurf-AI and ESM2-3B, ~11 GB). The first run takes a few minutes.

import os, sys, re, io, json, uuid, shutil, zipfile, datetime, contextlib, importlib, subprocess
import urllib.request, urllib.error, html as _html

IN_COLAB = "google.colab" in sys.modules


def _ensure(module, pip_name=None):
    try:
        return importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or module], check=True)
        importlib.invalidate_caches()
        return importlib.import_module(module)


for _mod, _pip_name in [("gemmi", "gemmi"), ("transformers", "transformers"), ("huggingface_hub", "huggingface_hub")]:
    _ensure(_mod, _pip_name)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import gemmi
import transformers
from IPython.display import display, HTML

# ── Configuration ────────────────────────────────────────────────────────────
# Direct download link to consurf_ai_3B_weights.pth, e.g. the file committed to the GitHub repository:
#   https://github.com/claudelelemaystdenis/<repo>/raw/main/<folder>/consurf_ai_3B_weights.pth
# (or a Zenodo / GitHub-release link). If left empty, the notebook asks the user to upload the file.
WEIGHTS_URL = "https://github.com/claudelelemaystdenis/ConSurf-AI/raw/main/weights/consurf_ai_3B_weights.pth
WEIGHTS_SHA256 = "99c06aa4494a361ecfc1ec306d32109bef1d9b80d1b78be70c64d046ff15205f"
ESM2_NAME = "facebook/esm2_t36_3B_UR50D"
MAX_LEN = 2000                       # the model's rotary-embedding table covers 2000 positions
WORK_DIR = "/content/consurf_ai" if IN_COLAB else os.path.abspath(os.environ.get("CONSURF_AI_WORKDIR", "consurf_ai_run"))
WEIGHTS_PATH = os.path.join(WORK_DIR, "weights", "consurf_ai_3B_weights.pth")
MOLSTAR_CDN = "https://cdn.jsdelivr.net/npm/molstar@5.13.1/build/viewer"
os.makedirs(WORK_DIR, exist_ok=True)

CONSURF_COLORS = {1: "#10C8D1", 2: "#8CFFFF", 3: "#D7FFFF", 4: "#EAFFFF", 5: "#FFFFFF",
                  6: "#FCEDF4", 7: "#FAC9DE", 8: "#F07DAB", 9: "#A02560"}
LOWCONF_COLOR = "#FFFF96"
OTHER_COLOR = "#C8C8C8"
LIGAND_CARBON = "#5CBF5C"
ELEMENT_COLORS = {"N": "#3050F8", "O": "#FF0D0D", "S": "#E6C200", "SE": "#FFA100", "P": "#FF8000",
                  "FE": "#E06633", "ZN": "#7D80B0", "MG": "#8AFF00", "CU": "#C88033"}

# ── Structure retrieval ──────────────────────────────────────────────────────
_PDB_RE = re.compile(r"^[0-9][A-Z0-9]{3}$")
_UNIPROT_RE = re.compile(r"^(?:[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9](?:[A-Z][A-Z0-9]{2}[0-9]){1,2})$")


def _open_url(url, timeout=300):
    req = urllib.request.Request(url, headers={"User-Agent": "ConSurf-AI-Colab"})
    return urllib.request.urlopen(req, timeout=timeout)


def _download(url, path):
    tmp = path + ".part"
    with _open_url(url) as r, open(tmp, "wb") as f:
        shutil.copyfileobj(r, f)
    os.replace(tmp, path)


def fetch_structure(query, out_dir):
    """PDB ID -> RCSB mmCIF (+ the PDB-format file when the entry has one); UniProt accession / AF model ID ->
    AlphaFold DB model (mmCIF + PDB). The files are kept exactly as downloaded.
    Returns (files, label, description) with files = {"cif": path, "pdb": path} (whichever exist)."""
    q = query.strip().upper()
    if not q:
        raise ValueError("Empty query: enter a PDB ID or UniProt accession, or tick 'upload_structure'.")
    af_id = None
    m = re.match(r"^AF-([A-Z0-9]+)-F(\d+)", q)
    if m:
        q, af_id = m.group(1), f"AF-{m.group(1)}-F{m.group(2)}"
    if _PDB_RE.match(q):
        files = {"cif": os.path.join(out_dir, f"{q}.cif")}
        try:
            _download(f"https://files.rcsb.org/download/{q}.cif", files["cif"])
        except urllib.error.HTTPError as e:
            raise ValueError(f"PDB entry '{q}' could not be downloaded from RCSB (HTTP {e.code}).") from None
        try:                                     # large entries have no PDB-format file
            pdb_path = os.path.join(out_dir, f"{q}.pdb")
            _download(f"https://files.rcsb.org/download/{q}.pdb", pdb_path)
            files["pdb"] = pdb_path
        except urllib.error.URLError:
            pass
        return files, q, f"PDB entry {q} (asymmetric unit, model 1)"
    if _UNIPROT_RE.match(q):
        try:
            with _open_url(f"https://alphafold.ebi.ac.uk/api/prediction/{q}", timeout=60) as r:
                entries = json.loads(r.read())
        except urllib.error.HTTPError as e:
            raise ValueError(f"No AlphaFold DB model found for UniProt accession '{q}' (HTTP {e.code}).") from None
        if not entries:
            raise ValueError(f"No AlphaFold DB model found for UniProt accession '{q}'.")
        if af_id:
            entries = [e for e in entries if (e.get("entryId") or e.get("modelEntityId") or "").upper() == af_id] or entries
        own = [e for e in entries if (e.get("uniprotAccession") or "").upper() == q]
        entry = (own or entries)[0]
        model_id = entry.get("entryId") or entry.get("modelEntityId") or f"AF-{q}-F1"
        files = {}
        for fmt, key in (("cif", "cifUrl"), ("pdb", "pdbUrl")):
            if entry.get(key):
                files[fmt] = os.path.join(out_dir, f"{model_id}.{fmt}")
                _download(entry[key], files[fmt])
        if not files:
            raise ValueError(f"AlphaFold DB returned no structure file for '{q}'.")
        desc = entry.get("uniprotDescription") or ""
        org = entry.get("organismScientificName") or ""
        return files, model_id, f"AlphaFold DB {model_id} – {desc} ({org})".replace(" ()", "")
    raise ValueError(f"'{query}' is neither a PDB ID (e.g. 4HHB) nor a UniProt accession (e.g. P69905).")


def structure_format(path):
    """'cif' for mmCIF files, 'pdb' otherwise (judged from the content, not the extension)."""
    with open(path, errors="replace") as f:
        head = f.read(4096)
    return "cif" if re.search(r"^\s*(data_|loop_|_atom_site\.)", head, re.M) else "pdb"


def load_structure(path):
    """Parsed copy used only to read the sequence and residue numbering. Output files and the 3D view
    are made from the original file, never from this object."""
    st = gemmi.read_structure(path)
    st.setup_entities()
    while len(st) > 1:                       # keep the first model only (NMR ensembles)
        del st[len(st) - 1]
    st.remove_alternative_conformations()    # keep the first conformer / first microheterogeneity residue
    return st


_ALLOWED_AA = set("ACDEFGHIKLMNPQRSTVWYXUOBZ")


def _residue_letter(res):
    info = gemmi.find_tabulated_residue(res.name)
    if info is None or not info.is_amino_acid():
        return "X"
    letter = info.one_letter_code.upper()     # lowercase = modified residue, e.g. MSE -> 'm' -> 'M'
    return letter if letter in _ALLOWED_AA else "X"


def _protein_polymer(chain):
    poly = chain.get_polymer()
    if len(poly) == 0:
        return None
    if poly.check_polymer_type() not in (gemmi.PolymerType.PeptideL, gemmi.PolymerType.PeptideD):
        return None
    return poly


def _res_key(r):
    return f"{r.seqid.num}{r.seqid.icode.strip()}"


def chain_residues(st, chain_name):
    """Observed residues of a protein chain: list of dicts (key, number, icode, resname, aa)."""
    for ch in st[0]:
        if ch.name != chain_name:
            continue
        poly = _protein_polymer(ch)
        if poly is None:
            continue
        return [{"key": _res_key(r), "number": r.seqid.num, "icode": r.seqid.icode.strip(),
                 "resname": r.name, "aa": _residue_letter(r)} for r in poly]
    return None


def list_protein_chains(st):
    rows, seen = [], set()
    for ch in st[0]:
        if ch.name in seen or _protein_polymer(ch) is None:
            continue
        seen.add(ch.name)
        res = chain_residues(st, ch.name)
        seq = "".join(r["aa"] for r in res)
        rows.append({"chain": ch.name, "residues": len(res),
                     "numbering": f"{res[0]['key']}–{res[-1]['key']}",
                     "sequence": seq if len(seq) <= 60 else seq[:57] + "..."})
    return rows


def numbering_gaps(residues):
    return [(a["key"], b["key"]) for a, b in zip(residues, residues[1:]) if b["number"] - a["number"] > 1]


# ── ConSurf-AI network (inference-only copy of step1.py's AttentionNetwork) ──
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, head_dim, max_len=MAX_LEN):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float() / head_dim))
        freqs = torch.einsum("i,j->ij", torch.arange(max_len).float(), inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :])
        self.register_buffer("sin", emb.sin()[None, None, :, :])

    def forward(self, q, k):
        L = q.size(-2)
        cos, sin = self.cos[:, :, :L, :], self.sin[:, :, :L, :]

        def rotate_half(x):
            x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
            return torch.cat((-x2, x1), dim=-1)

        return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.c_fc = nn.Linear(cfg["reduced_dim"], cfg["hidden_dimension"], bias=cfg["bias"])
        self.gelu = nn.GELU(approximate="none")
        self.c_proj = nn.Linear(cfg["hidden_dimension"], cfg["reduced_dim"], bias=cfg["bias"])

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))


class AttentionLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        D = cfg["reduced_dim"]
        self.num_heads = cfg["num_heads"]
        self.mlp = MLP(cfg)
        self.ln_1 = nn.LayerNorm(D, bias=cfg["bias"])
        self.ln_2 = nn.LayerNorm(D, bias=cfg["bias"])
        self.rope = RotaryPositionalEmbedding(D // self.num_heads, max_len=MAX_LEN)
        self.q_proj = nn.Linear(D, D, bias=cfg["bias"])
        self.k_proj = nn.Linear(D, D, bias=cfg["bias"])
        self.v_proj = nn.Linear(D, D, bias=cfg["bias"])
        self.out_proj = nn.Linear(D, D, bias=cfg["bias"])

    def forward(self, x, mask):
        h = self.ln_1(x)
        B, L, D = h.shape
        H = self.num_heads
        q = self.q_proj(h).view(B, L, H, D // H).transpose(1, 2)
        k = self.k_proj(h).view(B, L, H, D // H).transpose(1, 2)
        v = self.v_proj(h).view(B, L, H, D // H).transpose(1, 2)
        q, k = self.rope(q, k)
        attn_mask = mask.unsqueeze(1).unsqueeze(2) if mask is not None else None
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=0.0, is_causal=False)
        x = x + self.out_proj(out.transpose(1, 2).contiguous().view(B, L, D))
        return x + self.mlp(self.ln_2(x))


class AttentionNetwork(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.input_reduction = nn.Linear(cfg["inferred_input_dim"], cfg["reduced_dim"], bias=cfg["bias"])
        self.layers = nn.ModuleList([AttentionLayer(cfg) for _ in range(cfg["num_layers"])])
        self.ln_f = nn.LayerNorm(cfg["reduced_dim"], bias=cfg["bias"])
        self.lm_head = nn.Linear(cfg["reduced_dim"], 9, bias=cfg["bias"])
        self.conf_head = nn.Linear(cfg["reduced_dim"], 1, bias=cfg["bias"])

    def forward(self, x, mask):
        x = self.input_reduction(x)
        for layer in self.layers:
            x = layer(x, mask)
        x = self.ln_f(x)
        return self.lm_head(x), 8.0 * torch.sigmoid(self.conf_head(x).squeeze(-1))


# ── Weights download (cell 0) and model loading (cell 3, cached) ─────────────
def _sha256(path):
    import hashlib
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def download_consurf_ai_weights():
    if os.path.exists(WEIGHTS_PATH):
        return WEIGHTS_PATH
    os.makedirs(os.path.dirname(WEIGHTS_PATH), exist_ok=True)
    if WEIGHTS_URL and "<" not in WEIGHTS_URL:
        print("Downloading ConSurf-AI weights...")
        _download(WEIGHTS_URL, WEIGHTS_PATH)
        if WEIGHTS_SHA256 and _sha256(WEIGHTS_PATH) != WEIGHTS_SHA256:
            os.remove(WEIGHTS_PATH)
            raise RuntimeError(f"The file downloaded from {WEIGHTS_URL} is not the ConSurf-AI weights file "
                               "(checksum mismatch). Please report this issue.")
    elif IN_COLAB:
        from google.colab import files
        print("Please upload the ConSurf-AI weights file (consurf_ai_3B_weights.pth):")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No weights file uploaded.")
        with open(WEIGHTS_PATH, "wb") as f:
            f.write(next(iter(uploaded.values())))
    else:
        raise FileNotFoundError(f"Weights not found at {WEIGHTS_PATH} and WEIGHTS_URL is empty.")
    return WEIGHTS_PATH


def download_esm2(max_attempts=4):
    """Fetch the ESM2-3B PyTorch files into the Hugging Face cache (~11 GB; safetensors if the repo has them,
    otherwise the .bin shards; the TensorFlow copies are skipped).
    ESM2 is public, so no Hugging Face token is needed. A token (Colab 🔑 Secrets → HF_TOKEN) only
    raises the rate limit, so the 'unauthenticated requests' warning is hidden when none is set."""
    import time, logging, warnings
    from huggingface_hub import snapshot_download
    warnings.filterwarnings("ignore", message=".*unauthenticated requests.*")
    logging.getLogger("huggingface_hub.utils._http").setLevel(logging.ERROR)
    print(f"Downloading {ESM2_NAME} (~11 GB, first run only)...")
    for attempt in range(1, max_attempts + 1):
        try:
            path = snapshot_download(ESM2_NAME, allow_patterns=["*.json", "*.txt", "*.safetensors"])
            if not any(f.endswith(".safetensors") for f in os.listdir(path)):
                path = snapshot_download(ESM2_NAME, allow_patterns=["*.json", "*.txt", "*.bin"])
            return path
        except Exception as e:                       # network hiccup or HF rate limit (HTTP 429)
            if attempt == max_attempts:
                raise RuntimeError(
                    f"Could not download {ESM2_NAME} from Hugging Face ({type(e).__name__}: {e}).\n"
                    "If this is a rate-limit error (429), create a free token at https://huggingface.co/settings/tokens, "
                    "add it in Colab's 🔑 Secrets panel as HF_TOKEN (enable notebook access), then rerun cell 0."
                ) from None
            wait = 30 * attempt
            print(f"   Download interrupted ({type(e).__name__}); retrying in {wait} s "
                  f"(attempt {attempt + 1}/{max_attempts})...")
            time.sleep(wait)


_MODELS = {}


def load_models():
    if _MODELS:
        return _MODELS
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif os.environ.get("CONSURF_AI_ALLOW_CPU"):
        device = torch.device("cpu")
    else:
        raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → select a GPU (e.g. T4), "
                           "then run all cells again.")

    ckpt = torch.load(download_consurf_ai_weights(), map_location="cpu", weights_only=True)
    net = AttentionNetwork(ckpt["config"])
    state = {k.removeprefix("_orig_mod.").removeprefix("module."): v for k, v in ckpt["model"].items()}
    net.load_state_dict(state, strict=True)
    net.to(device).eval()

    from transformers import AutoTokenizer, EsmModel
    from transformers.utils import logging as hf_logging
    print("Loading ESM2-3B onto the GPU..." if device.type == "cuda" else "Loading ESM2-3B...")
    dtype = torch.float32
    if device.type == "cuda" and torch.cuda.get_device_properties(0).total_memory / 2**30 < 14:
        dtype = torch.float16                # fp32 ESM2-3B needs ~11.4 GB of weights + activations
    tv = tuple(int(x) for x in re.findall(r"\d+", transformers.__version__)[:2])
    kw = {"add_pooling_layer": False, ("dtype" if tv >= (4, 56) else "torch_dtype"): dtype}
    if tv < (5, 0):
        kw["low_cpu_mem_usage"] = True
    # Only the encoder is used (per-residue last-layer embeddings), so the masked-LM head stored in the
    # checkpoint is skipped on purpose; silence transformers' report about those unused weights.
    prev = hf_logging.get_verbosity()
    hf_logging.set_verbosity_error()
    hf_logging.disable_progress_bar()
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            tokenizer = AutoTokenizer.from_pretrained(ESM2_NAME)
            try:
                esm = EsmModel.from_pretrained(ESM2_NAME, device_map={"": 0}, **kw) if device.type == "cuda" \
                    else EsmModel.from_pretrained(ESM2_NAME, **kw)
            except (ImportError, ValueError):   # accelerate missing -> plain load
                esm = EsmModel.from_pretrained(ESM2_NAME, **kw).to(device)
    finally:
        hf_logging.set_verbosity(prev)
        hf_logging.enable_progress_bar()
    esm.eval()
    _MODELS.update(device=device, net=net, esm=esm, tokenizer=tokenizer, esm_dtype=dtype)
    print(f"Models ready on {device} (ESM2 in {str(dtype).replace('torch.', '')}).")
    return _MODELS


@torch.no_grad()
def predict_grades(sequence):
    """Returns grades (1-9), the 9 grade probabilities, the raw uncertainty (0-8, lower = better),
    the rounded uncertainty and the 'confident' flag (rounded uncertainty < 2), same conventions
    as the evaluation scripts."""
    m = load_models()
    dev = m["device"]
    tokens = m["tokenizer"](sequence, return_tensors="pt").to(dev)
    hidden = m["esm"](**tokens).last_hidden_state[0, 1:1 + len(sequence)].float()   # drop BOS/EOS
    if not torch.isfinite(hidden).all():
        raise RuntimeError("ESM2 produced non-finite embeddings (fp16 overflow?). Try a GPU with more memory.")
    mask = torch.ones(1, len(sequence), dtype=torch.bool, device=dev)
    logits, unc = m["net"](hidden.unsqueeze(0), mask)
    probs = F.softmax(logits[0].float(), dim=-1)
    grades = probs.argmax(dim=-1) + 1
    unc = unc[0].float()
    unc_rounded = torch.floor(unc + 0.5).clamp(0, 8).int()
    return (grades.cpu().numpy(), probs.cpu().numpy(), unc.cpu().numpy(),
            unc_rounded.cpu().numpy(), (unc_rounded < 2).cpu().numpy())


# ── Output files ─────────────────────────────────────────────────────────────
def _safe(name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", name).strip("_") or "query"


def grades_table(residues, chain, grades, probs, unc_raw, unc_rounded, confident):
    df = pd.DataFrame({
        "POS": np.arange(1, len(residues) + 1),
        "CHAIN": chain,
        "RESIDUE_NUMBER": [r["key"] for r in residues],
        "RESNAME": [r["resname"] for r in residues],
        "AA": [r["aa"] for r in residues],
        "GRADE": grades.astype(int),
        "UNCERTAINTY_RAW": np.round(unc_raw, 3),
        "UNCERTAINTY": unc_rounded.astype(int),
        "CONFIDENT": confident.astype(bool),
    })
    for g in range(9):
        df[f"P_GRADE_{g + 1}"] = np.round(probs[:, g], 4)
    return df


def _query_subchains(st, chain):
    for ch in st[0]:
        poly = _protein_polymer(ch) if ch.name == chain else None
        if poly is not None:
            return {r.subchain for r in poly}
    return set()


def cif_with_bfactors(path, out_path, chain, values_by_key, subchains, other=0.0):
    """Copy of an mmCIF file in which only _atom_site.B_iso_or_equiv is changed: values_by_key[residue] on the
    query chain and `other` everywhere else. Every other category (incl. _chem_comp_bond, _struct_conn) is kept."""
    doc = gemmi.cif.read(path)
    block = doc[0]
    tab = block.find("_atom_site.", ["auth_asym_id", "auth_seq_id", "?pdbx_PDB_ins_code", "?label_asym_id",
                                     "B_iso_or_equiv"])
    if not tab:
        raise ValueError(f"{path}: no _atom_site table with auth_asym_id / auth_seq_id / B_iso_or_equiv.")
    s = gemmi.cif.as_string
    for row in tab:
        v = None
        if s(row[0]) == chain and (not row.has(3) or s(row[3]) in subchains):
            icode = s(row[2]).strip() if row.has(2) and row[2] not in ("?", ".") else ""
            v = values_by_key.get(s(row[1]) + icode)
        row[4] = f"{float(v if v is not None else other):.2f}"
    doc.write_file(out_path)


def pdb_with_bfactors(path, out_path, chain, values_by_key, other=0.0):
    """Copy of a PDB file in which only columns 61-66 (B-factor) of ATOM/HETATM records are changed.
    All other records (HEADER, SEQRES, LINK, CONECT, ...) are copied unchanged."""
    with open(path, errors="replace") as fin, open(out_path, "w") as fout:
        for line in fin:
            if line.startswith(("ATOM  ", "HETATM")) and len(line.rstrip("\r\n")) >= 54:
                eol = line[len(line.rstrip("\r\n")):]
                body = line.rstrip("\r\n").ljust(66)
                v = values_by_key.get(body[22:26].strip() + body[26].strip()) if body[21] == chain else None
                line = body[:60] + f"{float(v if v is not None else other):6.2f}" + body[66:] + eol
            fout.write(line)


def _rgb01(hex_color):
    h = hex_color.lstrip("#")
    return [round(int(h[i:i + 2], 16) / 255, 3) for i in (0, 2, 4)]


def pymol_script(df, chain, structure_file):
    obj, q = "consurf_ai", f"consurf_ai and polymer and chain {chain}"
    lines = [f"# ConSurf-AI colouring. Run from the folder containing {structure_file}:",
             "#   pymol <this_script>.pml", f"load {structure_file}, {obj}",
             f"hide everything, {obj}", f"show cartoon, {obj}", f"color grey80, {obj}",
             f"show sticks, {obj} and organic", f"color green, {obj} and organic and elem C",
             f"color atomic, {obj} and organic and not elem C",
             f"show spheres, {obj} and inorganic", f"set sphere_scale, 0.5, {obj} and inorganic"]
    for g, c in CONSURF_COLORS.items():
        lines.append(f"set_color consurf_{g}, {_rgb01(c)}")
    lines.append(f"set_color consurf_lowconf, {_rgb01(LOWCONF_COLOR)}")
    for g in range(1, 10):
        keys = df.loc[(df.GRADE == g) & df.CONFIDENT, "RESIDUE_NUMBER"].tolist()
        if keys:
            lines.append(f"color consurf_{g}, {q} and resi {'+'.join(map(str, keys))}")
    low = df.loc[~df.CONFIDENT, "RESIDUE_NUMBER"].tolist()
    if low:
        lines.append(f"color consurf_lowconf, {q} and resi {'+'.join(map(str, low))}")
    lines += ["bg_color white", f"orient {q}"]
    return "\n".join(lines) + "\n"


def chimerax_script(df, chain, structure_file):
    lines = [f"# ConSurf-AI colouring for ChimeraX (keep next to {structure_file}); open this .cxc file in ChimeraX",
             f"open {structure_file}", "cartoon", "color #1 lightgray",
             "show ligand", "style ligand stick", "color ligand & C forestgreen", "color ligand byhetero"]
    for g, c in CONSURF_COLORS.items():
        keys = df.loc[(df.GRADE == g) & df.CONFIDENT, "RESIDUE_NUMBER"].tolist()
        if keys:
            lines.append(f"color #1/{chain}:{','.join(map(str, keys))} & protein {c}")
    low = df.loc[~df.CONFIDENT, "RESIDUE_NUMBER"].tolist()
    if low:
        lines.append(f"color #1/{chain}:{','.join(map(str, low))} & protein {LOWCONF_COLOR}")
    lines.append(f"view #1/{chain}")
    return "\n".join(lines) + "\n"


def write_outputs(state, df):
    out_dir = os.path.join(WORK_DIR, "results", f"{state['label']}_{state['chain']}")
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir)
    base = f"{state['label']}_{state['chain']}_ConSurf-AI"
    written = []

    csv_path = os.path.join(out_dir, f"{base}_grades.csv")
    with open(csv_path, "w") as f:
        f.write(f"# ConSurf-AI predictions for {state['source']}, chain {state['chain']}\n")
        f.write("# GRADE: 1 = most variable ... 9 = most conserved. UNCERTAINTY_RAW: predicted uncertainty "
                "(0-8, lower = better); UNCERTAINTY: rounded value; CONFIDENT = UNCERTAINTY < 2. "
                "P_GRADE_k: predicted probability of grade k.\n")
        df.to_csv(f, index=False)
    written.append(csv_path)

    # The structure files are the input files with only the B-factor column replaced (no re-writing,
    # so bonds, CONECT/LINK records, ligand definitions and all metadata are preserved).
    values = dict(zip(df.RESIDUE_NUMBER, df.GRADE.astype(int)))
    inputs, struct_name = state["inputs"], None
    if "cif" in inputs:
        cif_path = os.path.join(out_dir, f"{base}_structure.cif")
        cif_with_bfactors(inputs["cif"], cif_path, state["chain"], values,
                          _query_subchains(state["structure"], state["chain"]))
        written.append(cif_path)
        struct_name = os.path.basename(cif_path)
    if "pdb" in inputs:
        pdb_path = os.path.join(out_dir, f"{base}_structure.pdb")
        pdb_with_bfactors(inputs["pdb"], pdb_path, state["chain"], values)
        written.append(pdb_path)
        struct_name = os.path.basename(pdb_path)      # PDB preferred for the scripts (CONECT is read everywhere)

    for ext, fn in (("pml", pymol_script), ("cxc", chimerax_script)):
        p = os.path.join(out_dir, f"{base}_{'pymol' if ext == 'pml' else 'chimerax'}.{ext}")
        with open(p, "w") as f:
            f.write(fn(df, state["chain"], struct_name))
        written.append(p)
    return out_dir, written


# ── Visualisation (Mol* viewer driven by a MolViewSpec scene) ────────────────
_BACKBONE = {"N", "C", "O", "OXT"}


def legend_html():
    boxes = "".join(
        f'<div style="width:30px;height:22px;background:{CONSURF_COLORS[g]};border:1px solid #888;'
        f'display:flex;align-items:center;justify-content:center;color:{"#fff" if g == 9 else "#000"};'
        f'font:bold 12px sans-serif">{g}</div>' for g in range(1, 10))
    return (f'<div style="display:flex;align-items:center;gap:8px;font:13px sans-serif;margin:6px 0">'
            f'<span>Variable</span><div style="display:flex">{boxes}</div><span>Conserved</span>'
            f'<div style="width:30px;height:22px;background:{LOWCONF_COLOR};border:1px solid #888;margin-left:16px"></div>'
            f'<span>Uncertain (uncertainty ≥ 2)</span></div>')


def sequence_html(df, highlight_low_confidence=True, per_row=50):
    rows = []
    for start in range(0, len(df), per_row):
        chunk = df.iloc[start:start + per_row]
        cells = []
        for i, r in enumerate(chunk.itertuples()):
            low = highlight_low_confidence and not r.CONFIDENT
            bg = LOWCONF_COLOR if low else CONSURF_COLORS[r.GRADE]
            fg = "#fff" if (r.GRADE == 9 and not low) else "#000"
            tip = f"{r.RESNAME} {r.CHAIN}{r.RESIDUE_NUMBER} – grade {r.GRADE}, uncertainty {r.UNCERTAINTY_RAW:.2f}"
            sep = "margin-left:5px;" if i and i % 10 == 0 else ""
            cells.append(f'<span title="{_html.escape(tip)}" style="{sep}display:inline-block;width:15px;'
                         f'text-align:center;background:{bg};color:{fg};font:13px monospace;line-height:16px">'
                         f'{r.AA}<br><span style="font-size:9px">{r.GRADE}</span></span>')
        rows.append(f'<div style="white-space:nowrap;margin:2px 0"><span style="display:inline-block;width:52px;'
                    f'text-align:right;margin-right:6px;font:11px monospace;color:#888">{chunk.RESIDUE_NUMBER.iloc[0]}'
                    f'</span>{"".join(cells)}</div>')
    return f'<div style="overflow-x:auto;padding:4px 0;background:#fff">{"".join(rows)}</div>'


def _view_annotation(state, df, highlight_low_confidence):
    """MolViewSpec annotation (JSON rows keyed by author chain / residue number / insertion code)."""
    rows, rows_icode = [], []
    for res, r in zip(state["residues"], df.itertuples()):
        row = {"auth_asym_id": state["chain"], "auth_seq_id": int(res["number"]),
               "color": LOWCONF_COLOR if (highlight_low_confidence and not r.CONFIDENT) else CONSURF_COLORS[int(r.GRADE)],
               "tooltip": f"ConSurf-AI grade {r.GRADE} | uncertainty {r.UNCERTAINTY_RAW:.2f}"
                          + ("" if r.CONFIDENT else " (uncertain)")}
        if res["icode"]:
            row["pdbx_PDB_ins_code"] = res["icode"]
            rows_icode.append(row)
        else:
            rows.append(row)
    return rows + rows_icode           # rows with insertion codes last so they take precedence


def _mvs_scene(state, df, representation, show_other_chains, show_ligands, parse_format):
    chain = state["chain"]
    ann = {"uri": "__ANNOTATION_URL__", "format": "json", "schema": "auth_residue"}
    graded = [{"kind": "color_from_uri", "params": {**ann, "field_name": "color"}}]
    by_element = [{"kind": "color", "params": {"color": c, "selector": {"type_symbol": e}}}
                  for e, c in ELEMENT_COLORS.items()]
    no_h = {"ignore_hydrogens": True}
    rep_type, extra = {"surface": ("surface", no_h), "sticks": ("ball_and_stick", no_h)}.get(representation, ("cartoon", {}))
    comps = [{"kind": "component_from_uri", "params": {**ann, "field_name": "tooltip"}, "children": [
        {"kind": "representation", "params": {"type": rep_type, **extra}, "children": graded},
        {"kind": "focus", "params": {}}]}]
    min_grade = {"cartoon + sticks for grade 9": 9, "cartoon + sticks for grades 7-9": 7}.get(representation)
    if min_grade:
        atoms_by_key = {_res_key(r): [a.name for a in r if a.name not in _BACKBONE and a.element.name not in ("H", "D")]
                        for ch in state["structure"][0] if ch.name == chain and _protein_polymer(ch)
                        for r in ch.get_polymer()}
        res_by_key = {r["key"]: r for r in state["residues"]}
        sel = []
        for k in df.loc[(df.GRADE >= min_grade) & df.CONFIDENT, "RESIDUE_NUMBER"]:
            atoms, res = atoms_by_key.get(k, []), res_by_key[k]
            if len(atoms) > 1:                               # side chain incl. CA; Gly skipped
                base = {"auth_asym_id": chain, "auth_seq_id": int(res["number"])}
                if res["icode"]:
                    base["pdbx_PDB_ins_code"] = res["icode"]
                sel += [{**base, "auth_atom_id": a} for a in atoms]
        if sel:
            comps.append({"kind": "component", "params": {"selector": sel}, "children": [
                {"kind": "representation", "params": {"type": "ball_and_stick", **no_h},
                 "children": graded + by_element}]})
    if show_other_chains:
        others = sorted({ch.name for ch in state["structure"][0] if len(ch.get_polymer()) and ch.name != chain})
        if others:
            comps.append({"kind": "component", "params": {"selector": [{"auth_asym_id": s} for s in others]},
                          "children": [{"kind": "representation", "params": {"type": "cartoon"},
                                        "children": [{"kind": "color", "params": {"color": OTHER_COLOR}}]}]})
    if show_ligands:
        for sel, rtype, size in (("ligand", "ball_and_stick", 1), ("ion", "spacefill", 0.6)):
            comps.append({"kind": "component", "params": {"selector": sel}, "children": [
                {"kind": "representation", "params": {"type": rtype, "size_factor": size, **no_h},
                 "children": [{"kind": "color", "params": {"color": LIGAND_CARBON}}] + by_element}]})
    comps.append({"kind": "tooltip_from_uri", "params": {**ann, "field_name": "tooltip"}})
    return {"metadata": {"version": "1", "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat()},
            "root": {"kind": "root", "children": [
                {"kind": "canvas", "params": {"background_color": "white"}},
                {"kind": "download", "params": {"url": "__STRUCTURE_URL__"}, "children": [
                    {"kind": "parse", "params": {"format": parse_format}, "children": [
                        {"kind": "structure", "params": {"type": "model"}, "children": comps}]}]}]}}


def molstar_view_html(state, df, representation="cartoon", highlight_low_confidence=True,
                      show_other_chains=True, show_ligands=True, height=700):
    """The viewer loads the original structure file unchanged; ConSurf-AI colours and tooltips come from a
    separate annotation, so bonds and ligands are shown exactly as in the downloaded/uploaded file."""
    fmt = "cif" if "cif" in state["inputs"] else "pdb"
    with open(state["inputs"][fmt], errors="replace") as f:
        structure_text = f.read()
    annotation = _view_annotation(state, df, highlight_low_confidence)
    scene = _mvs_scene(state, df, representation, show_other_chains, show_ligands,
                       "mmcif" if fmt == "cif" else "pdb")
    eid = "consurf_ai_" + uuid.uuid4().hex[:10]
    return f"""
<link rel="stylesheet" href="{MOLSTAR_CDN}/molstar.css">
<style>#{eid} .msp-state-snapshot-viewport-controls, #{eid} .msp-animation-viewport-controls {{display:none !important}}</style>
<div id="{eid}" style="position:relative;width:100%;height:{height}px;"></div>
<script>
(function() {{
  const structure = {json.dumps(structure_text)};
  const annotation = {json.dumps(json.dumps(annotation))};
  const mvs = {json.dumps(json.dumps(scene))};
  function withMolstar(cb) {{
    if (window.molstar) return cb();
    const s = document.createElement('script');
    s.src = '{MOLSTAR_CDN}/molstar.js';
    s.onload = cb;
    s.onerror = () => {{ document.getElementById('{eid}').innerText = 'Could not load the Mol* viewer from the CDN.'; }};
    document.head.appendChild(s);
  }}
  withMolstar(async () => {{
    const url = URL.createObjectURL(new Blob([structure], {{type: 'text/plain'}})) + '#structure.{fmt}';
    const annUrl = URL.createObjectURL(new Blob([annotation], {{type: 'application/json'}})) + '#annotation.json';
    const viewer = await molstar.Viewer.create('{eid}', {{   // RCSB-style layout: sequence on top, controls on the right
      layoutIsExpanded: false, layoutShowControls: true, layoutShowRemoteState: false,
      layoutShowSequence: true, layoutShowLog: false, layoutShowLeftPanel: true, collapseLeftPanel: true,
      viewportShowAnimation: false, viewportShowTrajectoryControls: false,
      volumeStreamingDisabled: true, viewportBackgroundColor: '#ffffff',
    }});
    await viewer.loadMvsData(mvs.split('__STRUCTURE_URL__').join(url).split('__ANNOTATION_URL__').join(annUrl), 'mvsj');
    try {{   // thin sticks: atom spheres as wide as the bonds (no balls)
      const plugin = viewer.plugin, b = plugin.state.data.build();
      let n = 0;
      plugin.state.data.cells.forEach(cell => {{
        const p = cell.transform.params;
        if (p && p.type && p.type.name === 'ball-and-stick') {{
          b.to(cell.transform.ref).update(Object.assign({{}}, p, {{ type: {{ name: 'ball-and-stick',
            params: Object.assign({{}}, p.type.params, {{ sizeFactor: 0.12, sizeAspectRatio: 1 }}) }} }}));
          n++;
        }}
      }});
      if (n) await b.commit();
    }} catch (e) {{ console.warn('sticks', e); }}
    window['{eid}'] = viewer;
  }});
}})();
</script>"""


STATE = {}
CONSURF_AI_READY = True
print(f"✅ ConSurf-AI helpers ready (torch {torch.__version__}, transformers {transformers.__version__}, "
      f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none – please select a GPU runtime'})")

# ── Download model weights now so later cells only load them ────────────────
download_consurf_ai_weights()
print("✅ ConSurf-AI weights ready")
download_esm2()
print("✅ ESM2-3B weights ready")

In [ ]:
#@title 1. Provide a structure {display-mode: "form"}
#@markdown Enter a **PDB ID** (e.g. `4HHB`) or a **UniProt accession** (e.g. `P69905`; its model is fetched from **AlphaFold DB**). An AlphaFold model ID such as `AF-P69905-F1` also works.
#@markdown Or tick **upload_structure** to upload your own PDB / mmCIF file.
query = "4HHB"  #@param {type:"string"}
upload_structure = False  #@param {type:"boolean"}

if "CONSURF_AI_READY" not in globals():
    raise RuntimeError("Please run cell 0 first.")

_input_dir = os.path.join(WORK_DIR, "input")
shutil.rmtree(_input_dir, ignore_errors=True)
os.makedirs(_input_dir)

if upload_structure:
    if not IN_COLAB:
        raise RuntimeError("Uploading is only available in Google Colab.")
    from google.colab import files
    _uploaded = files.upload()
    if not _uploaded:
        raise RuntimeError("No file uploaded.")
    _fname, _data = next(iter(_uploaded.items()))
    if _fname.lower().endswith(".gz"):
        import gzip
        _data, _fname = gzip.decompress(_data), _fname[:-3]
    _path = os.path.join(_input_dir, os.path.basename(_fname))
    with open(_path, "wb") as f:
        f.write(_data)
    _files = {structure_format(_path): _path}
    _label = re.sub(r"\.(pdb|ent|cif|mmcif)$", "", os.path.basename(_fname), flags=re.I)
    _source = f"uploaded file {_fname}"
else:
    _files, _label, _source = fetch_structure(query, _input_dir)

_path = _files.get("cif") or _files["pdb"]
STATE = {"inputs": _files, "path": _path, "label": _safe(_label), "source": _source,
         "structure": load_structure(_path)}
STATE["chains"] = list_protein_chains(STATE["structure"])
if not STATE["chains"]:
    raise ValueError("No protein chain found in this structure.")

print(f"Loaded {_source}")
print(f"{len(STATE['chains'])} protein chain(s) found:")
display(pd.DataFrame(STATE["chains"]).set_index("chain"))

In [ ]:
#@title 2. Select a chain {display-mode: "form"}
#@markdown Type the chain ID from the table above. Leave it empty to use the first protein chain.
chain_id = "A"  #@param {type:"string"}

if not STATE.get("chains"):
    raise RuntimeError("Please run cell 1 first.")
_available = [c["chain"] for c in STATE["chains"]]
_chain = chain_id.strip() or _available[0]
if _chain not in _available:
    raise ValueError(f"Chain '{_chain}' not found. Available protein chains: {', '.join(_available)}")

_res = chain_residues(STATE["structure"], _chain)
_seq = "".join(r["aa"] for r in _res)
STATE.update(chain=_chain, residues=_res, sequence=_seq)
STATE.pop("results", None)

print(f"Chain {_chain}: {len(_seq)} residues ({_res[0]['key']}–{_res[-1]['key']})\n")
for i in range(0, len(_seq), 60):
    print(f"{_res[i]['key']:>6}  {_seq[i:i + 60]}")
if len(_seq) > MAX_LEN:
    print(f"\n⚠️ The chain is longer than {MAX_LEN} residues, which is the maximum ConSurf-AI supports.")
_gaps = numbering_gaps(_res)
if _gaps:
    print(f"\n⚠️ {len(_gaps)} gap(s) in the residue numbering (unmodelled residues), e.g. "
          + ", ".join(f"{a}→{b}" for a, b in _gaps[:5])
          + ". The model only sees residues present in the structure, so the fragments are joined.")
if _seq.count("X"):
    print(f"\nNote: {_seq.count('X')} non-standard/unknown residue(s) are encoded as X.")

In [ ]:
#@title 3. Run ConSurf-AI {display-mode: "form"}
#@markdown The first run loads ESM2-3B onto the GPU (~1 min). Later runs reuse it and take about a second per chain.
import time

if not STATE.get("sequence"):
    raise RuntimeError("Please run cells 1 and 2 first.")
if len(STATE["sequence"]) > MAX_LEN:
    raise ValueError(f"Chain {STATE['chain']} has {len(STATE['sequence'])} residues; ConSurf-AI supports up to {MAX_LEN}.")

load_models()
_t0 = time.time()
_df = grades_table(STATE["residues"], STATE["chain"], *predict_grades(STATE["sequence"]))
_out_dir, _written = write_outputs(STATE, _df)
STATE.update(results=_df, out_dir=_out_dir, files=_written)

print(f"✅ Predicted {len(_df)} residues of chain {STATE['chain']} in {time.time() - _t0:.1f} s")
print(f"   Uncertain residues (uncertainty ≥ 2): {(~_df.CONFIDENT).sum()} ({(~_df.CONFIDENT).mean():.0%})")
_counts = _df.GRADE.value_counts().reindex(range(1, 10), fill_value=0)
print("   Grade distribution: " + "  ".join(f"{g}:{n}" for g, n in _counts.items()))
print(f"\nFiles written to {_out_dir}:")
for f in _written:
    print("   " + os.path.basename(f))
display(_df.iloc[:, :9].head(10))

In [ ]:
#@title 4. Visualise {display-mode: "form"}
#@markdown Mol* viewer: hover over a residue to see its grade and uncertainty. Other chains are grey and ligands have green carbons.
representation = "cartoon"  #@param ["cartoon", "cartoon + sticks for grade 9", "cartoon + sticks for grades 7-9", "surface", "sticks"]
highlight_low_confidence = True  #@param {type:"boolean"}
show_other_chains = True  #@param {type:"boolean"}
show_ligands = True  #@param {type:"boolean"}

if "results" not in STATE:
    raise RuntimeError("Please run cell 3 first.")
display(HTML(f"<h3 style='font-family:sans-serif;margin:4px 0'>ConSurf-AI – {_html.escape(STATE['label'])} "
             f"chain {_html.escape(STATE['chain'])}</h3>" + legend_html()))
display(HTML(molstar_view_html(STATE, STATE["results"], representation, highlight_low_confidence,
                               show_other_chains, show_ligands)))
display(HTML(sequence_html(STATE["results"], highlight_low_confidence)))

In [ ]:
#@title 5. Download results {display-mode: "form"}
#@markdown Downloads a zip with the per-residue grades (CSV), the structure with grades in the B-factor column (PDB/mmCIF) and PyMOL / ChimeraX colouring scripts.
if "results" not in STATE:
    raise RuntimeError("Please run cell 3 first.")
_zip_path = os.path.join(WORK_DIR, f"{STATE['label']}_{STATE['chain']}_ConSurf-AI.zip")
with zipfile.ZipFile(_zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(STATE["out_dir"])):
        z.write(os.path.join(STATE["out_dir"], f), arcname=f)
print(f"Created {_zip_path}")
if IN_COLAB:
    from google.colab import files
    files.download(_zip_path)

### Output files
| File | Content |
|---|---|
| `*_grades.csv` | One row per residue: chain numbering, amino acid, **grade (1–9)**, **uncertainty** (raw 0–8 value and rounded; lower = better), confidence flag (rounded uncertainty < 2) and the probability of each of the 9 grades |
| `*_structure.pdb` / `.cif` | The input structure file, unchanged except for the **B-factor** column, which holds the grade on the query chain (0 elsewhere). All other records (bonds, CONECT/LINK, ligand definitions, metadata) are kept as in the original file |
| `*_pymol.pml` | PyMOL script that colours the structure in ConSurf colours and shows ligands as sticks (`pymol script.pml`, run next to the structure file) |
| `*_chimerax.cxc` | The same colouring for ChimeraX (open the .cxc file next to the structure) |

**Notes.** Predictions use the residues present in the structure (first model, first alternate conformation). Missing residues are not filled in from SEQRES. Chains of up to 2000 residues are supported.